In [2]:
import os
import sys
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings,ChatOllama
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print(f"Python version: {sys.version}")
print("ALL imports successful!")
print("Ready for the offline RAG")

Python version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
ALL imports successful!
Ready for the offline RAG


In [3]:
!ollama list

NAME                        ID              SIZE      MODIFIED     
nomic-embed-text:latest     0a109f422b47    274 MB    7 months ago    
gemma3:270m                 e7d36fb2c3b3    291 MB    7 months ago    
mxbai-embed-large:latest    468836162de7    669 MB    7 months ago    


In [7]:
print("Test Ollama connection\n")

try:
    test_llm = ChatOllama(model="gemma3:270m",temperature=0)
    response = test_llm.invoke("Say 'Hello!I am running locally on your machine'")
    print(f"Response: {response.content}")
except Exception as e:
    print(f"Error connecting to Ollama {e}")

Test Ollama connection

Response: Hello! I am running locally on your machine. 



In [9]:
pdf_path="./Content/resume.pdf"

if not os.path.exists(pdf_path):
  print(f"Pdf file {pdf_path} not found!")
else:
  loader = PyPDFLoader(pdf_path)
  documents = loader.load()

  print(f"Loaded document from {pdf_path}")
  print("First page preview")
  print(f"Content(first 200 chars) {documents[0].page_content[:200]}")
  print(f"Metadata: {documents[0].metadata}")
  print(f"Total chars in the document:{sum(len(doc.page_content) for doc in documents):,}")

Loaded document from ./Content/resume.pdf
First page preview
Content(first 200 chars) Naveen Manjappa 
Experienced in designing and implementing SaaS-based warehouse management solutions 
(SCALE), I bridge business needs with technology through full-stack development (Angular, 
.NET 
A
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-07-12T20:39:02+01:00', 'author': 'Naveen Kengannar Manjappa', 'moddate': '2026-07-12T20:39:02+01:00', 'source': './Content/resume.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}
Total chars in the document:3,861


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1024,
  chunk_overlap=128,
  length_function=len,
  separators=["\n\n","\n"," ",""]
)

chunks = text_splitter.split_documents(documents)

avg_chunk_size = sum(len(chunk.page_content) for chunk in chunks)/len(chunks)
print(f"Split {len(documents)} documents into {len(chunks)} chunks")

for i,chunk in enumerate(chunks[:3],1):
  print(f"Chunk {i} (length: {len(chunk.page_content)})")
  print(f"{chunk.page_content[:100]}")

Split 4 documents into 54 chunks
Chunk 1 (length: 15)
Naveen Manjappa
Chunk 2 (length: 83)
Experienced in designing and implementing SaaS-based warehouse management solutions
Chunk 3 (length: 95)
(SCALE), I bridge business needs with technology through full-stack development (Angular, 
.NET
Chunk 4 (length: 90)
.NET 
APIs, SQL Server, HTML, JS, CSS,Bootstrap,Azure functions) and process optimization.
Chunk 5 (length: 94)
Skilled in cloud 
based technical solution development and architecture, I drive smooth system


In [14]:
#Create embeddings
embeddings = OllamaEmbeddings(
  model="nomic-embed-text:latest"
)

sample_text="This is a test sentence for embeddings"
sample_emebddings = embeddings.embed_query(sample_text)

print(f"Embedding dimension: {len(sample_emebddings)}")
print(f"Sample embedding(first 10 chars) {sample_emebddings[:10]}")


Embedding dimension: 768
Sample embedding(first 10 chars) [0.025940226, 0.06043353, -0.16587152, -0.076955654, 0.03810934, -0.030227156, 0.050242305, -0.017312462, -0.015698105, -0.031052535]


In [15]:
#Create Chroma DB vector store
persistent_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
  documents=chunks,
  embedding=embeddings,
  persist_directory=persistent_directory,
  collection_name="local_rag_collection"
)

print(f"Indexed {len(chunks)} chunks")
print(f"Vector store persisted to disk at {pdf_path}")


Indexed 54 chunks
Vector store persisted to disk at ./Content/resume.pdf


In [16]:
#Retreiver test
retreiver = vectorstore.as_retriever(
  search_type="similarity",
  search_kwargs={"k":4}
)

search_query = "What are the interests of Naveen as mentioned in the document?"
retreived_docs = retreiver.invoke(search_query)
print(f"Retreived {len(retreived_docs)} documents")

for i,doc in enumerate(retreived_docs,1):
  print(f"Document {i}")
  print(f"Content preview: {doc.page_content[:120]}")
  print(f"Source: {doc.metadata}")


Retreived 4 documents
Document 1
Content preview: Naveen Manjappa
Source: {'total_pages': 4, 'source': './Content/resume.pdf', 'author': 'Naveen Kengannar Manjappa', 'producer': 'Microsoft® Word for Microsoft 365', 'page_label': '1', 'moddate': '2026-07-12T20:39:02+01:00', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-07-12T20:39:02+01:00', 'page': 0}
Document 2
Content preview: Developed a deeper interest in the business side of things, focusing on
Source: {'creator': 'Microsoft® Word for Microsoft 365', 'author': 'Naveen Kengannar Manjappa', 'source': './Content/resume.pdf', 'moddate': '2026-07-12T20:39:02+01:00', 'page': 1, 'page_label': '2', 'creationdate': '2026-07-12T20:39:02+01:00', 'total_pages': 4, 'producer': 'Microsoft® Word for Microsoft 365'}
Document 3
Content preview: Interests 
Investing - Reading - Farming
Source: {'source': './Content/resume.pdf', 'page_label': '4', 'creator': 'Microsoft® Word for Microsoft 365', 'author': 'Naveen Kengannar Man

In [17]:
llm = ChatOllama(
  model="gemma3:270m",
  temperature=0
)

test_response = llm.invoke("What is the capital of France")
print(test_response.content)

The capital of France is Paris.



In [18]:
#Build RAG chain
system_prompt =(
  "You are a helpful assistant for question-answering tasks. "
  "Use the following pieces of retreived context to answer the question. "
  "If can't find the answer based on the context, say that you don't know. "
  "Keep the answer concise and accurate.\n\n"
  "Context:{context}"
  "Question:{question}"
)

prompt = ChatPromptTemplate.from_template(system_prompt)

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain =(
  {
    "context":retreiver | format_docs,
    "question":RunnablePassthrough()
  }
  | prompt
  | llm
  | StrOutputParser()
)

query1="What is the summary of this document?"
answer=rag_chain.invoke(query1)
print(answer)




The document describes a solution to extend the base product's functionality with custom technical extensions.

